# Agent 2 — Create Databricks Workflow

Creates the automated reconciliation workflow.
Schedule: Daily at 08:00 AM UTC (2 hours after Agent 1).

In [0]:
import requests
import json
import time

host  = spark.conf.get("spark.databricks.workspaceUrl")
token = (
    dbutils.notebook.entry_point
           .getDbutils().notebook().getContext()
           .apiToken().get()
)

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type" : "application/json"
}

EXISTING_CLUSTER_ID = "0220-160255-4reafcow"

ROOT      = "/Users/s819534@8451.com/data_sentinel_barrows"
PIPE_PATH = f"{ROOT}/agent2/pipeline"

RECONCILIATION_PATH = f"{PIPE_PATH}/01_reconciliation_agent"
AI_SUMM_PATH        = f"{PIPE_PATH}/02_ai_summarization"
ALERT_DISP_PATH     = f"{PIPE_PATH}/03_alert_dispatcher"

NOTIFICATION_EMAIL  = "s819534@8451.com"

print("=" * 60)
print("AGENT 2 — WORKFLOW CREATION")
print("=" * 60)
print(f"Cluster ID : {EXISTING_CLUSTER_ID}")
print(f"Paths:")
print(f"  01: {RECONCILIATION_PATH}")
print(f"  02: {AI_SUMM_PATH}")
print(f"  03: {ALERT_DISP_PATH}")
print("=" * 60)

## Step 1: Verify All Notebooks Exist

In [0]:
def verify_notebook(path: str) -> bool:
    response = requests.get(
        f"https://{host}/api/2.0/workspace/get-status",
        headers=headers,
        params={"path": path}
    )
    exists = response.status_code == 200
    status = "EXISTS  ✅" if exists else "MISSING ❌"
    print(f"  {status}  {path}")
    return exists

print("Verifying notebooks...")
print()

results = [
    verify_notebook(RECONCILIATION_PATH),
    verify_notebook(AI_SUMM_PATH),
    verify_notebook(ALERT_DISP_PATH)
]

print()

if not all(results):
    raise Exception(
        "One or more notebooks are missing. "
        "Create all three before creating the workflow."
    )

print("All notebooks verified.")

## Step 2: Create the Workflow

In [0]:
workflow = {
    "name": "agent2_cross_layer_reconciliation",

    "schedule": {
        "quartz_cron_expression": "0 0 8 * * ?",
        "timezone_id"           : "UTC",
        "pause_status"          : "PAUSED"
    },

    "email_notifications": {
        "on_failure"               : [NOTIFICATION_EMAIL],
        "on_success"               : [],
        "no_alert_for_skipped_runs": True
    },

    "max_concurrent_runs": 1,
    "timeout_seconds"    : 7200,

    "tags": {
        "team"   : "data-engineering",
        "project": "data-trust-layer",
        "agent"  : "agent2"
    },

    # ── Job Parameters ─────────────────────────────────────────────────────
    # Single source of truth for target catalog/schema. Exposed to every
    # notebook task below as widgets ("catalog" / "schema"), so promoting
    # the whole job to a new catalog/schema only requires editing these
    # two defaults instead of every notebook.
    "parameters": [
        {"name": "catalog", "default": "learning_dev"},
        {"name": "schema"  , "default": "s819534"}
    ],

    "tasks": [

        # ── Task 1: Reconciliation Agent ───────────────────────────────────
        {
            "task_key"           : "run_reconciliation_agent",
            "description"        : (
                "Runs cross-layer reconciliation checks "
                "across Raw, Silver, and Gold layers."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "notebook_task"      : {
                "notebook_path"   : RECONCILIATION_PATH,
                "base_parameters" : {
                    "business_date"      : "",
                    "source_table_filter": ""
                },
                "source"          : "WORKSPACE"
            },
            "timeout_seconds"          : 3600,
            "max_retries"              : 1,
            "min_retry_interval_millis": 300000,
            "retry_on_timeout"         : False,
            "email_notifications"      : {
                "on_failure": [NOTIFICATION_EMAIL]
            }
        },

        # ── Task 2: AI Summarization ───────────────────────────────────────
        {
            "task_key"           : "run_ai_summarization",
            "description"        : (
                "Enriches PENDING reconciliation runs "
                "with AI cross-layer drift explanations."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "depends_on"         : [
                {"task_key": "run_reconciliation_agent"}
            ],
            "notebook_task"      : {
                "notebook_path": AI_SUMM_PATH,
                "source"       : "WORKSPACE"
            },
            "timeout_seconds"          : 1800,
            "max_retries"              : 1,
            "min_retry_interval_millis": 120000,
            "retry_on_timeout"         : False
        },

        # ── Task 3: Alert Dispatcher ───────────────────────────────────────
        {
            "task_key"           : "run_alert_dispatcher",
            "description"        : (
                "Sends Teams alerts for CRITICAL and WARNING "
                "reconciliation runs."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "depends_on"         : [
                {"task_key": "run_ai_summarization"}
            ],
            "notebook_task"      : {
                "notebook_path": ALERT_DISP_PATH,
                "source"       : "WORKSPACE"
            },
            "timeout_seconds"          : 900,
            "max_retries"              : 2,
            "min_retry_interval_millis": 60000,
            "retry_on_timeout"         : False
        }
    ]
}

print("Creating workflow...")
print()

response = requests.post(
    f"https://{host}/api/2.1/jobs/create",
    headers=headers,
    json=workflow
)

print(f"Status : {response.status_code}")

if response.status_code == 200:
    job_id = response.json().get("job_id")
    print(f"\n✅ Workflow created.")
    print(f"   Job ID  : {job_id}")
    print(f"   View at : https://{host}/#job/{job_id}")
    print()
    print("Schedule is PAUSED. Run test before enabling.")
else:
    print(f"\n❌ Failed: {response.text}")
    job_id = None

## Step 3: Reset Test Data

In [0]:
print("Resetting 2026-06-24 for workflow test...")
print()

spark.sql("""
    DELETE FROM learning_dev.s819534.uc2_reconciliation_results_b
    WHERE business_date = '2026-06-24'
""")

spark.sql("""
    DELETE FROM learning_dev.s819534.uc2_reconciliation_run_b
    WHERE business_date = '2026-06-24'
""")

counts = spark.sql("""
    SELECT
        'results' AS tbl, COUNT(*) AS cnt
    FROM learning_dev.s819534.uc2_reconciliation_results_b
    WHERE business_date = '2026-06-24'
    UNION ALL
    SELECT
        'run'     AS tbl, COUNT(*) AS cnt
    FROM learning_dev.s819534.uc2_reconciliation_run_b
    WHERE business_date = '2026-06-24'
""").collect()

for row in counts:
    print(f"  {row['tbl']}: {row['cnt']} records (should be 0)")

## Step 4: Trigger Manual Test Run

In [0]:
if not job_id:
    print("No job_id. Run Step 2 first.")
else:
    TEST_DATE = "2026-06-24"

    trigger_payload = {
        "job_id"         : int(job_id),
        "notebook_params": {
            "business_date"      : TEST_DATE,
            "source_table_filter": ""
        }
    }

    print(f"Triggering test run for {TEST_DATE}...")

    trigger_response = requests.post(
        f"https://{host}/api/2.1/jobs/run-now",
        headers=headers,
        json=trigger_payload
    )

    print(f"Status : {trigger_response.status_code}")

    if trigger_response.status_code == 200:
        run_id = trigger_response.json().get("run_id")
        print(f"\n✅ Test run triggered.")
        print(f"   Run ID  : {run_id}")
        print(f"   View at : https://{host}/#job/{job_id}/run/{run_id}")
    else:
        print(f"\n❌ Trigger failed: {trigger_response.text}")
        run_id = None

## Step 5: Monitor the Run

In [0]:
def monitor_run(run_id: int, poll_seconds: int = 20):
    if not run_id:
        print("No run_id available.")
        return

    terminal_states = {"TERMINATED", "SKIPPED", "INTERNAL_ERROR"}

    task_emojis = {
        "run_reconciliation_agent": "🔍",
        "run_ai_summarization"    : "🤖",
        "run_alert_dispatcher"    : "📣"
    }

    status_emojis = {
        "RUNNING"        : "⏳",
        "TERMINATED"     : "✅",
        "INTERNAL_ERROR" : "❌",
        "PENDING"        : "⏸",
        "SKIPPED"        : "⏭"
    }

    print(f"Monitoring run {run_id}...")
    print()

    previous_states = {}

    while True:
        resp = requests.get(
            f"https://{host}/api/2.1/jobs/runs/get",
            headers=headers,
            params={"run_id": run_id}
        )

        if resp.status_code != 200:
            print(f"Polling error: {resp.status_code}")
            break

        run_data   = resp.json()
        state      = run_data.get("state", {})
        life_state = state.get("life_cycle_state", "UNKNOWN")
        result     = state.get("result_state", "")
        tasks      = run_data.get("tasks", [])

        from datetime import datetime as dt
        ts = dt.now().strftime("%H:%M:%S")
        print(f"[{ts}] Overall: {life_state} {result}")

        for task in tasks:
            task_key    = task.get("task_key", "")
            task_state  = task.get("state", {})
            task_life   = task_state.get("life_cycle_state", "PENDING")
            task_result = task_state.get("result_state", "")
            t_emoji     = task_emojis.get(task_key, "⚙️")
            s_emoji     = status_emojis.get(task_life, "❓")

            current = f"{task_life} {task_result}"
            if previous_states.get(task_key) != current:
                previous_states[task_key] = current
                print(
                    f"  {t_emoji} {task_key:<35} "
                    f"{s_emoji} {task_life:<15} "
                    f"{task_result}"
                )

        print()

        if life_state in terminal_states:
            print("=" * 50)
            if result == "SUCCESS":
                print("✅ Workflow completed successfully.")
            else:
                print(f"❌ Workflow ended: {result}")
                msg = state.get("state_message", "")
                if msg:
                    print(f"   Message: {msg}")
            print("=" * 50)
            break

        time.sleep(poll_seconds)


if run_id:
    monitor_run(int(run_id))

## Step 6: Verify Results

In [0]:
print("Results after workflow run:\n")

spark.sql("""
    SELECT
        business_date,
        SUBSTRING_INDEX(source_fully_qualified_name, '.', -1) AS table_name,
        overall_status,
        total_checks_passed,
        total_checks_warned,
        total_checks_critical,
        ai_enrichment_status,
        CASE WHEN alert_sent = true THEN 'YES' ELSE 'NO' END AS alert_fired,
        alert_channel
    FROM learning_dev.s819534.uc2_reconciliation_run_b
    WHERE business_date = '2026-06-24'
    ORDER BY source_fully_qualified_name
""").show(truncate=False)

## Step 7: Enable Schedule

Run this cell ONLY after a successful test run.

In [0]:
def enable_schedule(job_id: int):
    response = requests.patch(
        f"https://{host}/api/2.1/jobs/{job_id}",
        headers=headers,
        json={
            "new_settings": {
                "schedule": {
                    "quartz_cron_expression": "0 0 8 * * ?",
                    "timezone_id"           : "UTC",
                    "pause_status"          : "UNPAUSED"
                }
            }
        }
    )
    if response.status_code == 200:
        print(f"✅ Schedule enabled.")
        print(f"   Agent 2 will run daily at 08:00 AM UTC.")
        print(f"   https://{host}/#job/{job_id}")
    else:
        print(f"❌ Failed: {response.text}")


# Uncomment after successful test
# enable_schedule(int(job_id))
print("Schedule is currently PAUSED.")
print("Uncomment enable_schedule() after successful test.")